## **Trade data Preprocessing**

In [2]:
# Trade preprocessing

import pandas as pd
import os
import glob


save_dir = (
    "/lakehouse/default/Files/data/"
    "processed/trades"
)

os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(
    save_dir,
    "trade_processed.csv"
)


def process_trade_files(files, source):

    data = []

    for file in files:

        try:

            df = pd.read_csv(file)

            df.columns = (
                df.columns
                .str.strip()
            )

            df = df.rename(columns={
                "Date": "date",
                "Symbol": "symbol",
                "Open": "open",
                "High": "high",
                "Low": "low",
                "Close": "close",
                "Volume": "volume"
            })

            keep_cols = [
                "date",
                "symbol",
                "open",
                "high",
                "low",
                "close",
                "volume"
            ]

            df = df[keep_cols]

            df["date"] = pd.to_datetime(
                df["date"],
                errors="coerce"
            )

            df["source"] = source

            data.append(df)

        except Exception as e:

            print(
                f"ERROR -> "
                f"{os.path.basename(file)} : {e}"
            )

    return data


# ---------- first build ----------
if not os.path.exists(save_path):

    all_data = []

    # NSE historical
    all_data.extend(
        process_trade_files(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "nse_trade_hist/*.csv"
            ),
            "NSE"
        )
    )

    # NSE incremental
    all_data.extend(
        process_trade_files(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "nse_trade_inc/*.csv"
            ),
            "NSE"
        )
    )

    # BSE historical
    all_data.extend(
        process_trade_files(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "bse_trade_hist/*.csv"
            ),
            "BSE"
        )
    )

    # BSE incremental
    all_data.extend(
        process_trade_files(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "bse_trade_inc/*.csv"
            ),
            "BSE"
        )
    )

    final_df = pd.concat(
        all_data,
        ignore_index=True
    )

# ---------- daily update ----------
else:

    processed_df = pd.read_csv(
        save_path,
        parse_dates=["date"]
    )

    all_data = []

    # only incremental files
    all_data.extend(
        process_trade_files(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "nse_trade_inc/*.csv"
            ),
            "NSE"
        )
    )

    all_data.extend(
        process_trade_files(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "bse_trade_inc/*.csv"
            ),
            "BSE"
        )
    )

    new_df = pd.concat(
        all_data,
        ignore_index=True
    )

    final_df = pd.concat(
        [processed_df, new_df],
        ignore_index=True
    )


# clean
final_df = final_df.dropna(
    subset=[
        "open",
        "high",
        "low",
        "close"
    ]
)

before_rows = len(final_df)

final_df = final_df[
    (final_df["open"] > 0) &
    (final_df["high"] > 0) &
    (final_df["low"] > 0) &
    (final_df["close"] > 0)
]

print(
    "Bad rows removed:",
    before_rows - len(final_df)
)

# dedupe
final_df = final_df.drop_duplicates()

# sort
final_df = final_df.sort_values(
    ["date", "symbol"]
)

# save
final_df.to_csv(
    save_path,
    index=False
)

print("SUCCESS")
print("Rows:", len(final_df))
print("Saved:", save_path)

Bad rows removed: 17036
SUCCESS
Rows: 21241171
Saved: /lakehouse/default/Files/data/processed/trades/trade_processed.csv


In [2]:
df = pd.read_csv(
    "/lakehouse/default/Files/data/processed/trades/trade_processed.csv"
)

print(df.head())
print(df.columns.tolist())
print(df["source"].value_counts())
print(df.isnull().sum())

StatementMeta(, 6c046dea-79f6-492f-97bb-3cffe3fb3bf4, 5, Finished, Available, Finished, False)

         date      symbol        open        high         low       close  \
0  2001-01-01      1STCUS   50.801350   50.801350   50.801350   50.801350   
1  2001-01-01  3BBLACKBIO    9.981130    9.981130    9.981130    9.981130   
2  2001-01-01     3MINDIA  586.437439  586.437439  586.437439  586.437439   
3  2001-01-01     63MOONS   48.178387   49.554912   46.801861   48.178387   
4  2001-01-01  AARTIDRUGS    1.553383    1.553383    1.553383    1.553383   

   volume source  
0       0    BSE  
1       0    BSE  
2       0    BSE  
3     200    BSE  
4       0    BSE  
['date', 'symbol', 'open', 'high', 'low', 'close', 'volume', 'source']
source
BSE    14044299
NSE     7160893
Name: count, dtype: int64
date      0
symbol    0
open      0
high      0
low       0
close     0
volume    0
source    0
dtype: int64


## **Bulk Deals Preprocessing**

In [9]:
import glob
import pandas as pd
import os

files = (
    glob.glob("/lakehouse/default/Files/data/raw/nse_bulk_hist/*.csv")
    + glob.glob("/lakehouse/default/Files/data/raw/nse_bulk_inc/*.csv")
    + glob.glob("/lakehouse/default/Files/data/raw/bse_bulk_hist/*.csv")
    + glob.glob("/lakehouse/default/Files/data/raw/bse_bulk_inc/*.csv")
)

print("Total files:", len(files))

for file in files:

    try:
        df = pd.read_csv(file)

        print(
            os.path.basename(file),
            "-> rows:",
            len(df),
            "cols:",
            df.columns.tolist()
        )

    except Exception as e:

        print(
            os.path.basename(file),
            "ERROR ->",
            type(e).__name__,
            str(e)
        )

Total files: 28
bulkdeals_01-06-2026.csv -> rows: 228356 cols: ['"Date "', 'Symbol', 'Security Name', 'Client Name', 'Buy / Sell', 'Quantity Traded', 'Trade Price / Wght. Avg. Price', 'Remarks']
bulkdeals_02-06-2026.csv -> rows: 0 cols: ['Date', 'Symbol', 'Security Name', 'Client Name', 'Buy / Sell', 'Quantity Traded', 'Trade Price / Wght. Avg. Price', 'Remarks']
bulkdeals_03-06-2026.csv -> rows: 0 cols: ['Date', 'Symbol', 'Security Name', 'Client Name', 'Buy / Sell', 'Quantity Traded', 'Trade Price / Wght. Avg. Price', 'Remarks']
bulkdeals_04-06-2026.csv -> rows: 84 cols: ['Date', 'Symbol', 'Security Name', 'Client Name', 'Buy / Sell', 'Quantity Traded', 'Trade Price / Wght. Avg. Price', 'Remarks']
bulkdeals_05-06-2026.csv -> rows: 134 cols: ['Date', 'Symbol', 'Security Name', 'Client Name', 'Buy / Sell', 'Quantity Traded', 'Trade Price / Wght. Avg. Price', 'Remarks']
bulkdeals_06-06-2026.csv -> rows: 0 cols: ['Date', 'Symbol', 'Security Name', 'Client Name', 'Buy / Sell', 'Quantity T

In [ ]:
# Bulk preprocessing

import pandas as pd
import os
import glob


save_path = (
    "/lakehouse/default/Files/data/"
    "processed/deals/bulk_processed.csv"
)


def clean_bulk(files, source):

    data = []

    for file in files:

        # skip broken/empty files
        try:
            df = pd.read_csv(file)

        except pd.errors.EmptyDataError:
            print(
                "Skipped empty file:",
                os.path.basename(file)
            )
            continue

        # skip 0-row files
        if df.empty:
            continue

        # clean column names
        df.columns = (
            df.columns
            .str.strip()
            .str.replace('"', '', regex=False)
            .str.replace('\ufeff', '', regex=False)
        )

        # fix weird NSE column
        df.columns = [
            col.replace("Date ", "Date")
            for col in df.columns
        ]

        # NSE schema
        if source == "NSE":

            df = df.rename(columns={
                "Date": "date",
                "Symbol": "symbol",
                "Security Name": "security_name",
                "Client Name": "client_name",
                "Buy / Sell": "deal_type",
                "Quantity Traded": "quantity",
                "Trade Price / Wght. Avg. Price": "price"
            })

        # BSE schema
        else:

            df = df.rename(columns={
                "Deal Date": "date",
                "Deal_Date": "date",
                "Security Code": "symbol",
                "Security_Code": "symbol",
                "Company": "security_name",
                "Client Name": "client_name",
                "Client_Name": "client_name",
                "Deal Type": "deal_type",
                "Deal_Type": "deal_type",
                "Quantity": "quantity",
                "Price": "price"
            })

        keep_cols = [
            "date",
            "symbol",
            "security_name",
            "client_name",
            "deal_type",
            "quantity",
            "price"
        ]

        df = df[keep_cols]

        # trim text columns
        text_cols = [
            "symbol",
            "security_name",
            "client_name",
            "deal_type"
        ]

        for col in text_cols:
            df[col] = (
            df[col]
            .astype(str)
            .str.strip()
        )

        # quantity cleanup
        df["quantity"] = (
            df["quantity"]
            .astype(str)
            .str.replace(",", "", regex=False)
        )

        df["quantity"] = pd.to_numeric(
            df["quantity"],
            errors="coerce"
        )

        # price cleanup
        df["price"] = pd.to_numeric(
            df["price"],
            errors="coerce"
        )

        # standardize deal type
        df["deal_type"] = (
            df["deal_type"]
            .astype(str)
            .str.upper()
            .replace({
                "B": "BUY",
                "P": "BUY",
                "BUY": "BUY",
                "S": "SELL",
                "SELL": "SELL",
                "SOLD": "SELL"
            })
        )

        # standardize date
        df["date"] = pd.to_datetime(
            df["date"],
            errors="coerce",
            dayfirst=True
        )

        df["source"] = source

        data.append(df)

    # safety
    if len(data) == 0:
        return pd.DataFrame()

    return pd.concat(
        data,
        ignore_index=True
    )


# ---------- first run ----------
if not os.path.exists(save_path):

    final_df = pd.concat([

        clean_bulk(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "nse_bulk_hist/*.csv"
            ),
            "NSE"
        ),

        clean_bulk(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "nse_bulk_inc/*.csv"
            ),
            "NSE"
        ),

        clean_bulk(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "bse_bulk_hist/*.csv"
            ),
            "BSE"
        ),

        clean_bulk(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "bse_bulk_inc/*.csv"
            ),
            "BSE"
        )

    ], ignore_index=True)

# ---------- later runs ----------
else:

    processed_df = pd.read_csv(save_path)

    new_df = pd.concat([

        clean_bulk(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "nse_bulk_inc/*.csv"
            ),
            "NSE"
        ),

        clean_bulk(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "bse_bulk_inc/*.csv"
            ),
            "BSE"
        )

    ], ignore_index=True)

    final_df = pd.concat(
        [processed_df, new_df],
        ignore_index=True
    )


final_df = final_df.dropna(
    subset=[
        "date",
        "security_name",
        "deal_type"
    ]
)

final_df = final_df.drop_duplicates()

# remove invalid zero prices
final_df = final_df[
    (final_df["price"].isna()) |
    (final_df["price"] > 0)
]

final_df = final_df.sort_values(
    ["date", "symbol", "client_name"]
)

final_df.to_csv(
    save_path,
    index=False
)

print("SUCCESS")
print("Rows:", len(final_df))
print("Saved:", save_path)

/tmp/ipykernel_366/1468467227.py:140: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(


SUCCESS
Rows: 515492
Saved: /lakehouse/default/Files/data/processed/deals/bulk_processed.csv


## **Block Deals Preprocessing**

In [2]:
import pandas as pd
import glob
import os


files = (
    glob.glob(
        "/lakehouse/default/Files/data/raw/"
        "nse_block_hist/*.csv"
    )
    +
    glob.glob(
        "/lakehouse/default/Files/data/raw/"
        "nse_block_inc/*.csv"
    )
    +
    glob.glob(
        "/lakehouse/default/Files/data/raw/"
        "bse_block_hist/*.csv"
    )
    +
    glob.glob(
        "/lakehouse/default/Files/data/raw/"
        "bse_block_inc/*.csv"
    )
)

print("Total files:", len(files))

for file in files:

    try:

        df = pd.read_csv(file)

        print(
            "\n",
            os.path.basename(file)
        )

        print(
            "Rows:",
            len(df)
        )

        print(
            "Columns:",
            df.columns.tolist()
        )

    except Exception as e:

        print(
            os.path.basename(file),
            "ERROR ->",
            str(e)
        )

Total files: 6

 blockdeals_31-05-2026.csv
Rows: 12286
Columns: ['Date', 'Symbol', 'SecurityName', 'ClientName', 'Buy/Sell', 'QuantityTraded', 'TradePrice/Wght.Avg.Price', 'Remarks']

 blockdeals_02-06-2026.csv
Rows: 0
Columns: ['Date', 'Symbol', 'Security Name', 'Client Name', 'Buy / Sell', 'Quantity Traded', 'Trade Price / Wght. Avg. Price', 'Remarks']

 blockdeals_29-05-2026.csv
Rows: 29
Columns: ['Date', 'Symbol', 'Security Name', 'Client Name', 'Buy / Sell', 'Quantity Traded', 'Trade Price / Wght. Avg. Price', 'Remarks']

 blockdeals_01-06-2026.csv
Rows: 21240
Columns: ['Deal Date', 'Security Code', 'Company', 'Client Name', 'Deal Type', 'Quantity', 'Price']
block_02-06-2026.csv ERROR -> No columns to parse from file

 block_03-06-2026.csv
Rows: 2
Columns: ['Deal_Date', 'Security_Code', 'Company', 'Client_Name', 'Deal_Type', 'Quantity', 'Price']


In [3]:
# Block preprocessing

import pandas as pd
import os
import glob


save_path = (
    "/lakehouse/default/Files/data/"
    "processed/deals/block_processed.csv"
)


def clean_block(files, source):

    data = []

    for file in files:

        # skip empty files
        try:
            df = pd.read_csv(file)

        except pd.errors.EmptyDataError:

            print(
                "Skipped empty file:",
                os.path.basename(file)
            )
            continue

        # skip 0-row files
        if df.empty:
            continue

        # clean column names
        df.columns = (
            df.columns
            .str.strip()
            .str.replace('"', '', regex=False)
            .str.replace('\ufeff', '', regex=False)
        )

        # NSE schema
        if source == "NSE":

            df = df.rename(columns={

                # date
                "Date": "date",

                # common
                "Symbol": "symbol",

                # historical
                "SecurityName": "security_name",
                "ClientName": "client_name",
                "Buy/Sell": "deal_type",
                "QuantityTraded": "quantity",
                "TradePrice/Wght.Avg.Price": "price",

                # incremental
                "Security Name": "security_name",
                "Client Name": "client_name",
                "Buy / Sell": "deal_type",
                "Quantity Traded": "quantity",
                "Trade Price / Wght. Avg. Price": "price"
            })

        # BSE schema
        else:

            df = df.rename(columns={
                "Deal Date": "date",
                "Deal_Date": "date",
                "Security Code": "symbol",
                "Security_Code": "symbol",
                "Company": "security_name",
                "Client Name": "client_name",
                "Client_Name": "client_name",
                "Deal Type": "deal_type",
                "Deal_Type": "deal_type",
                "Quantity": "quantity",
                "Price": "price"
            })

        keep_cols = [
            "date",
            "symbol",
            "security_name",
            "client_name",
            "deal_type",
            "quantity",
            "price"
        ]

        df = df[keep_cols]

        # clean quantity
        df["quantity"] = pd.to_numeric(
            df["quantity"]
            .astype(str)
            .str.replace(",", "", regex=False),
            errors="coerce"
        )

        # clean price
        df["price"] = pd.to_numeric(
            df["price"],
            errors="coerce"
        )

        # standardize deal type
        df["deal_type"] = (
            df["deal_type"]
            .astype(str)
            .str.upper()
            .replace({
                "B": "BUY",
                "P": "BUY",
                "BUY": "BUY",
                "S": "SELL",
                "SELL": "SELL"
            })
        )

        # standardize date
        df["date"] = pd.to_datetime(
            df["date"],
            errors="coerce",
            dayfirst=True
        )

        df["source"] = source

        data.append(df)

    if len(data) == 0:
        return pd.DataFrame()

    return pd.concat(
        data,
        ignore_index=True
    )


# ---------- first run ----------
if not os.path.exists(save_path):

    final_df = pd.concat([

        clean_block(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "nse_block_hist/*.csv"
            ),
            "NSE"
        ),

        clean_block(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "nse_block_inc/*.csv"
            ),
            "NSE"
        ),

        clean_block(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "bse_block_hist/*.csv"
            ),
            "BSE"
        ),

        clean_block(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "bse_block_inc/*.csv"
            ),
            "BSE"
        )

    ], ignore_index=True)

# ---------- later runs ----------
else:

    processed_df = pd.read_csv(save_path)

    new_df = pd.concat([

        clean_block(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "nse_block_inc/*.csv"
            ),
            "NSE"
        ),

        clean_block(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "bse_block_inc/*.csv"
            ),
            "BSE"
        )

    ], ignore_index=True)

    final_df = pd.concat(
        [processed_df, new_df],
        ignore_index=True
    )


final_df = final_df.dropna(
    subset=[
        "date",
        "security_name",
        "deal_type"
    ]
)

final_df = final_df.drop_duplicates()

final_df = final_df.sort_values(
    ["date", "security_name"]
)

final_df.to_csv(
    save_path,
    index=False
)

print("SUCCESS")
print("Rows:", len(final_df))
print("Saved:", save_path)

Skipped empty file: block_02-06-2026.csv
SUCCESS
Rows: 14471
Saved: /lakehouse/default/Files/data/processed/deals/block_processed.csv


## **Securities Preprocessing**

In [1]:
# Security preprocessing

import pandas as pd
import os
import glob


# paths
save_dir = (
    "/lakehouse/default/Files/data/"
    "processed/securities"
)

os.makedirs(
    save_dir,
    exist_ok=True
)

save_path = os.path.join(
    save_dir,
    "security_processed.csv"
)


def clean_security(files, source):

    data = []

    for file in files:

        df = pd.read_csv(file)

        # clean columns
        df.columns = (
            df.columns
            .str.strip()
        )

        # ---------- NSE ----------
        if source == "NSE":

            df = df.rename(columns={
                "SYMBOL": "symbol",
                "NAME OF COMPANY": "company_name",
                "SERIES": "series",
                "DATE OF LISTING": "listing_date",
                "MARKET LOT": "market_lot",
                "FACE VALUE": "face_value",
                "ISIN NUMBER": "isin"
            })

            df = df[
                [
                    "symbol",
                    "company_name",
                    "series",
                    "listing_date",
                    "market_lot",
                    "face_value",
                    "isin"
                ]
            ]

            df["bse_security_code"] = None
            df["status"] = "Active"
            df["group"] = None
            df["instrument"] = "Equity"

        # ---------- BSE ----------
        else:

            df = df.rename(columns={
                "Security Code": "bse_security_code",
                "Issuer Name": "company_name",
                "Security Id": "symbol",
                "Status": "status",
                "Group": "group",
                "Face Value": "face_value",
                "ISIN No": "isin",
                "Instrument": "instrument"
            })

            df = df[
                [
                    "symbol",
                    "company_name",
                    "bse_security_code",
                    "status",
                    "group",
                    "face_value",
                    "isin",
                    "instrument"
                ]
            ]

            df["series"] = "EQ"
            df["listing_date"] = None
            df["market_lot"] = 1

        # clean listing date
        df["listing_date"] = pd.to_datetime(
            df["listing_date"],
            format="%d-%b-%Y",
            errors="coerce"
        )

        # numeric cleanup
        df["face_value"] = pd.to_numeric(
            df["face_value"],
            errors="coerce"
        )

        df["market_lot"] = pd.to_numeric(
            df["market_lot"],
            errors="coerce"
        )

        df["source"] = source

        data.append(df)

    if len(data) == 0:
        return pd.DataFrame()

    return pd.concat(
        data,
        ignore_index=True
    )


# ---------- first run ----------
if not os.path.exists(save_path):

    final_df = pd.concat([

        clean_security(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "nse_security_hist/*.csv"
            ),
            "NSE"
        ),

        clean_security(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "nse_security_inc/*.csv"
            ),
            "NSE"
        ),

        clean_security(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "bse_security_hist/*.csv"
            ),
            "BSE"
        ),

        clean_security(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "bse_security_inc/*.csv"
            ),
            "BSE"
        )

    ], ignore_index=True)

# ---------- later runs ----------
else:

    processed_df = pd.read_csv(
        save_path
    )

    new_df = pd.concat([

        clean_security(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "nse_security_inc/*.csv"
            ),
            "NSE"
        ),

        clean_security(
            glob.glob(
                "/lakehouse/default/Files/data/raw/"
                "bse_security_inc/*.csv"
            ),
            "BSE"
        )

    ], ignore_index=True)

    final_df = pd.concat(
        [processed_df, new_df],
        ignore_index=True
    )


# clean
final_df = final_df.dropna(
    subset=[
        "symbol",
        "company_name",
        "isin"
    ]
)

# remove duplicates
final_df = final_df.drop_duplicates()

# sort
final_df = final_df.sort_values(
    ["symbol"]
)

# save
final_df.to_csv(
    save_path,
    index=False
)

print("SUCCESS")
print("Rows:", len(final_df))
print("Saved:", save_path)

SUCCESS
Rows: 7917
Saved: /lakehouse/default/Files/data/processed/securities/security_processed.csv


In [4]:
# Check if the missing BSE code is present
print(df[df["bse_security_code"] == 538897])

# Check by symbol
print(df[df["symbol"] == "SHRINIWAS"])

# Check by company name
print(
    df[df["company_name"].str.contains("SHRINIWAS", case=False, na=False)]
)

# Total rows
print("Total rows:", len(df))

         symbol                            company_name series listing_date  \
6324  SHRINIWAS  Shri Niwas Leasing and Finance Limited     EQ          NaN   
6325  SHRINIWAS  Shri Niwas Leasing and Finance Limited     EQ          NaN   

      market_lot  face_value          isin  bse_security_code  status group  \
6324           1        10.0  INE201F01015           538897.0  Active    X    
6325           1        10.0  INE201F01015           538897.0  Active    XT   

     instrument source  
6324     Equity    BSE  
6325     Equity    BSE  
         symbol                            company_name series listing_date  \
6324  SHRINIWAS  Shri Niwas Leasing and Finance Limited     EQ          NaN   
6325  SHRINIWAS  Shri Niwas Leasing and Finance Limited     EQ          NaN   

      market_lot  face_value          isin  bse_security_code  status group  \
6324           1        10.0  INE201F01015           538897.0  Active    X    
6325           1        10.0  INE201F01015           

In [3]:
import pandas as pd

df = pd.read_csv(
    "/lakehouse/default/Files/data/"
    "processed/securities/security_processed.csv"
)

print(df.head())
print(df.columns.tolist())
print(df["source"].value_counts())
print(df.isnull().sum())

  symbol              company_name series listing_date  market_lot  \
0  08ABB  Nippon India Mutual Fund     EQ          NaN           1   
1  08ADD  Nippon India Mutual Fund     EQ          NaN           1   
2  08ADR  Nippon India Mutual Fund     EQ          NaN           1   
3  08AGG  Nippon India Mutual Fund     EQ          NaN           1   
4  08AMD  Nippon India Mutual Fund     EQ          NaN           1   

   face_value          isin  bse_security_code  status group instrument source  
0        10.0  INF204KB17R6           543151.0  Active    B      Equity    BSE  
1        10.0  INF204KB11S7           543170.0  Active    B      Equity    BSE  
2        10.0  INF204KB12S5           543145.0  Active    B      Equity    BSE  
3        10.0  INF204KB18R4           543153.0  Active    B      Equity    BSE  
4        10.0  INF204KB13S3           543147.0  Active    B      Equity    BSE  
['symbol', 'company_name', 'series', 'listing_date', 'market_lot', 'face_value', 'isin', 'bse